# L9 · Game Theory for Executives

**Análise Prescritiva — Camada de Alfabetização de Dados (Learning)**

| Campo | Detalhe |
|---|---|
| **Notebook** | L9 · Game Theory for Executives |
| **Autor** | Matheus Mendes |
| **Data** | 27/julho/2026 |
| **Versão** | 1.0 |
| **Público-alvo** | Executivos não-técnicos - "e se o concorrente?" |
| **Dependências** | numpy, plotly, json |
| **Fonte quantitativa** | NB-04 · Teoria dos Jogos EV Brasil (5 jogadores × 2 estratégias, 32 células) |

---

## Por que este notebook existe

Em **L0** você aprendeu que a média não conta a história inteira. Em **L1** viu que a incerteza tem forma - uma distribuição, não um número. Em **L4** acompanhou o caminho de uma única variável ao longo do tempo. Em **L6** cruzou quatro variáveis em 10.000 cenários. Em **L7** descobriu que oito variáveis se organizam em três drivers. Em **L8** encontrou o tamanho certo do hedge ótimo dentro de uma cerca de restrições.

Em **L9** subimos um nível de abstração. A pergunta deixa de ser "quanto hedge?"/"quanto alocar?"/"quanto buffer?" e passa a ser "o que o concorrente vai fazer - e o que faço quando ele faz?". Esta pergunta é o território da **teoria dos jogos não-cooperativos**.

A decisão executiva em jogo não é só matemática. É sobre **antecipar** o que BYD, Tesla, VW e demais jogadores farão, identificar o **equilíbrio de Nash** (o ponto onde ninguém pode melhorar sozinho), e posicionar a empresa em uma estratégia defensável. No caso BYD Camaçari 2025-2027, cinco jogadores disputam 500 mil unidades/ano de mercado EV; cada um escolhe entre **DIFFERENTIATE** (margem 8%) e **PRICE_WAR** (guerra de preços); e o equilíbrio encontrado em NB-04 aponta para uma estratégia de **diferenciação** para BYD.

Este caderno é **narrativa-primeiro**:

> Conceito -> Intuição -> Matemática -> Código -> Recado Executivo

### A pergunta central em uma frase

**"Dado que cinco fabricantes disputam o mercado EV brasileiro com duas estratégias (diferenciação vs guerra de preços), qual é o equilíbrio estável - e como a BYD deve se posicionar nele?"** A resposta é o Equilíbrio de Nash: BYD = DIFFERENTIATE, Stellantis = PRICE_WAR, GM = PRICE_WAR, VW = PRICE_WAR, Geely = DIFFERENTIATE. R$ 5,52 bi de valor agregado - e a frase executiva é: **"Nossa estratégia competitiva é diferenciação"**.

### Os 6 conceitos deste caderno

| # | Conceito | Pergunta executiva |
|---|---|---|
| 1 | Decisões estratégicas | Por que não basta fazer a matemática? |
| 2 | Jogadores e estratégias | Quem decide o quê, e em que espaço? |
| 3 | Payoffs | Como medir o valor de cada combinação? |
| 4 | Equilíbrio de Nash | O que é um resultado estável? |
| 5 | Competição | BYD vs Tesla vs VW - quem faz o quê? |
| 6 | Executivo: diferenciação | Como resumir tudo em uma frase? |


In [1]:
import json
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

NOTEBOOK_ROOT = Path.cwd()
NB04_PATH_CANDIDATES = [
    NOTEBOOK_ROOT / "outputs" / "nb04_results.json",
    NOTEBOOK_ROOT.parent / "outputs" / "nb04_results.json",
    Path(r"C:\\Users\\mathe\\code_space\\orchestration\\value-factory\\case-studies\\byd-camacari-2025-2027\\analise-prescritiva\\outputs") / "nb04_results.json",
]
NB04_PATH = None
for candidate in NB04_PATH_CANDIDATES:
    if candidate.exists():
        NB04_PATH = candidate
        break
if NB04_PATH is None:
    raise FileNotFoundError(f"nb04_results.json não encontrado em {NB04_PATH_CANDIDATES}")
OUT_DIR = NOTEBOOK_ROOT.parent / "outputs" / "learning"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with NB04_PATH.open(encoding="utf-8") as f:
    NB04 = json.load(f)

BG, INK, MUTED, GRID = "#0d1117", "#e8edf5", "#9baabb", "#30363d"
AZUL, TIJOLO, TEAL = "#0284c7", "#dc2626", "#0d9488"
VIOLETA, AMBAR, VERDE = "#9333ea", "#ea580c", "#22c55e"

PLAYERS = [p["name"] for p in NB04["players"]]
STRATEGIES = NB04["strategies"]
NASH = NB04["nash_equilibrium"]
AGG_DIFF = NB04["aggregate_npv_all_differentiate_B"]
AGG_WAR = NB04["aggregate_npv_all_war_B"]
AGG_NASH = NB04["aggregate_npv_at_nash_B"]
DELTA_NASH = NB04["value_of_nash_vs_war_B"]

def style(fig, title, height=440):
    fig.update_layout(
        template="plotly_dark", paper_bgcolor=BG, plot_bgcolor=BG,
        title=dict(text=title, x=0.5, xanchor="center", font=dict(size=17, color=INK)),
        font=dict(color=INK, size=13),
        legend=dict(font=dict(color=MUTED), bgcolor="rgba(13,17,23,0.6)",
                    bordercolor=GRID, borderwidth=1),
        margin=dict(l=65, r=40, t=75, b=55), height=height,
    )
    fig.update_xaxes(color=MUTED, gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID)
    fig.update_yaxes(color=MUTED, gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID)
    return fig

print("NB-04 carregado - 5 jogadores, 32 celulas, Nash E3 verificado:")
print(f"  Jogadores     : {PLAYERS}")
print(f"  Estrategias   : {STRATEGIES}")
print(f"  Nash E3       : {NASH}")
print(f"  Payoff NPV (R$ bi) - todos DIFFERENTIATE: {AGG_DIFF:.2f}")
print(f"  Payoff NPV (R$ bi) - todos PRICE_WAR    : {AGG_WAR:.2f}")
print(f"  Payoff NPV (R$ bi) - Nash E3            : {AGG_NASH:.2f}")
print(f"  Delta (Nash - War)                      : R$ {DELTA_NASH:.2f} bi")


NB-04 carregado - 5 jogadores, 32 celulas, Nash E3 verificado:
  Jogadores     : ['BYD', 'Stellantis', 'GM', 'VW', 'Geely']
  Estrategias   : ['DIFFERENTIATE', 'PRICE_WAR']
  Nash E3       : {'BYD': 'DIFFERENTIATE', 'Stellantis': 'PRICE_WAR', 'GM': 'PRICE_WAR', 'VW': 'PRICE_WAR', 'Geely': 'DIFFERENTIATE'}
  Payoff NPV (R$ bi) - todos DIFFERENTIATE: 19.20
  Payoff NPV (R$ bi) - todos PRICE_WAR    : -5.52
  Payoff NPV (R$ bi) - Nash E3            : 5.52
  Delta (Nash - War)                      : R$ 11.04 bi


## 1 - Decisoes estrategicas: nao e so matematica

Quando o CFO pergunta "e se o concorrente?", o analista que so sabe fazer matematica responde: *"o modelo preve 7,3 bi para BYD em Nash"*. E uma resposta correta, mas incompleta. **A decisao estrategica** e mais do que um payoff: e sobre escolher um **posicionamento** que funcione quando o outro jogador escolhe algo diferente do que voce imaginou. E sobre construir uma **resposta defensavel** para inumeras contingencies, nao para um unico cenario.

A diferenca para **otimizacao** (L8) e sutil mas decisiva: em L8, o analista maximiza uma funcao; aqui, ele escolhe uma acao sabendo que o valor dela muda quando os outros escolhem. A **estabilidade** da escolha (nenhum jogador se arrepende) e tao importante quanto o seu payoff.

### Recado executivo

- "7,3 bi" e a resposta do NB-04; "como chegar la e como defender" e o trabalho de L9.
- Estrategia nao e planilha, e posicao. Diferenciacao e guerra de precos nao sao colunas em uma matriz; sao **posicionamentos de mercado** com historia, marca e consequencias.
- O executivo pensa 3-5 lances a frente; o otimizador pensa 1.


In [2]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "L8 - Otimizacao: maximizar U(h)",
        "L9 - Estrategia: posicionar entre 32 perfis",
    ),
    horizontal_spacing=0.13,
)

h = np.linspace(0.0, 1.0, 200)
u_l8 = (1.0 - np.exp(-3.0 * h)) - 0.5 * (2.10 * (1.0 - h**1.5)) - 0.10 * h
fig.add_scatter(x=h, y=u_l8, mode="lines",
                line=dict(color=AZUL, width=3),
                name="U(h) - L8", showlegend=False, row=1, col=1)
idx_l8 = int(np.argmax(u_l8))
fig.add_scatter(x=[h[idx_l8]], y=[u_l8[idx_l8]], mode="markers+text",
                marker=dict(size=12, color=AMBAR, symbol="star",
                            line=dict(color=INK, width=1.2)),
                text=[f" h* = {h[idx_l8]:.0%}"], textposition="top right",
                textfont=dict(color=AMBAR, size=11), row=1, col=1,
                showlegend=False)

n_cells = 32
np.random.seed(42)
theta = np.linspace(0, 2 * np.pi, n_cells, endpoint=False)
x_ring = np.cos(theta) + np.random.normal(0, 0.05, n_cells)
y_ring = np.sin(theta) + np.random.normal(0, 0.05, n_cells)
colors_ring = []
for i in range(n_cells):
    byd_idx = (i >> 0) & 1
    if byd_idx == 0:
        colors_ring.append(VERDE)
    else:
        colors_ring.append(TIJOLO)
fig.add_scatter(x=x_ring, y=y_ring, mode="markers",
                marker=dict(size=12, color=colors_ring,
                            line=dict(color=INK, width=1.0)),
                name="32 perfis", row=1, col=2, showlegend=False)

nash_pos = (np.cos(0), np.sin(0))
fig.add_scatter(x=[nash_pos[0]], y=[nash_pos[1]], mode="markers+text",
                marker=dict(size=22, color=AMBAR, symbol="star",
                            line=dict(color=INK, width=2)),
                text=[" Nash E3"], textposition="top right",
                textfont=dict(color=AMBAR, size=12), row=1, col=2,
                showlegend=False)

style(fig, "1 - L8 otimiza 1 decisao; L9 posiciona entre 32 perfis", height=460)
fig.update_xaxes(title="h - hedge", row=1, col=1)
fig.update_yaxes(title="U(h) - utilidade", row=1, col=1)
fig.update_xaxes(title="perfil 1 (proj. PCA)", row=1, col=2,
                 showgrid=False, zeroline=False, scaleanchor="y")
fig.update_yaxes(title="perfil 2 (proj. PCA)", row=1, col=2,
                 showgrid=False, zeroline=False)
fig.write_html(str(OUT_DIR / "l9-01-strategic-vs-optimization.html"),
               include_plotlyjs="cdn", full_html=True)
fig.show()

print("L8 - Otimizacao: busca 1 numero otimo dentro de restricoes")
print("L9 - Estrategia: busca posicionamento estavel em 32 perfis")
print("Diferenca-chave: L8 ignora o concorrente; L9 espera reacao.")


L8 - Otimizacao: busca 1 numero otimo dentro de restricoes
L9 - Estrategia: busca posicionamento estavel em 32 perfis
Diferenca-chave: L8 ignora o concorrente; L9 espera reacao.
